In [1]:
# from canvas import learn_game, play_game, human_interaction, move_arrowkeys, move_regression, move_classification
from torch_bc import AgentNetwork_Regression
import pickle
import pandas as pd
import random
import matplotlib.pyplot as plt
from canvas import play_game

from torchvision import transforms
from PIL import Image
import numpy as np

pygame 2.6.1 (SDL 2.28.4, Python 3.12.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
def preprocess_to_movement(action):
    ## (right - left, up - down) for (dx, dy)
    
    return (int(action[3]) - int(action[2]), int(action[0]) - int(action[1]))
  
def print_full(x):
    pd.set_option('display.max_rows', len(x))
    print(x)
    pd.reset_option('display.max_rows')
    
    
folder = "screenshots-obs-seq-1k-withcols-closeness"
# folder = "test"
df = pd.read_pickle(f'./datasets/{folder}/ss-info.pkl')
print(f"df shape: {df.shape}")

# df.iloc[122]
df
# df[df.index.str.contains("close")]
# fracDf = df.sample(frac=0.5)
# print(f"fracDf shape: {fracDf.shape}")

df shape: (190045, 6)


,agent_x,agent_y,target_x,target_y,action,closeness
ss-0-close-0-seq-0.png,460,319,375,250,"(-5, -5)",0
ss-0-close-0-seq-1.png,455,314,375,250,"(-5, -5)",0
ss-0-close-0-seq-2.png,450,309,375,250,"(-5, -5)",0
ss-0-close-0-seq-3.png,445,304,375,250,"(-5, -5)",0
ss-0-close-0-seq-4.png,440,299,375,250,"(-5, -5)",0
...,...,...,...,...,...,...
ss-999-close-5-seq-109.png,161,421,91,420,"(-5, 0)",5
ss-999-close-5-seq-110.png,156,421,91,420,"(-5, 0)",5
ss-999-close-5-seq-111.png,151,421,91,420,"(-5, 0)",5
ss-999-close-5-seq-112.png,146,421,91,420,"(-5, 0)",5


In [6]:
import re
df["number"] = df.index.map(lambda s: int(re.search(r"ss-(\d+)-close-(\d+)-seq-(\d+).png", s)[1]))
df
# s = "ss-1-close-2-seq-3.png"
# cs = re.search(r"ss-(\d+)-close-(\d+)-seq-(\d+).png", s)
# cs[0]

closeness = {
      0: 0.001,
      1: 0.001,
      2: 0.001,
      3: 0.001,
      4: 0.001, ## 25
      5: 0.001, ##35
    }
sampledIndices = {c : np.random.choice(np.arange(0, 1000), size=int(frac * 1000)) for c, frac in closeness.items()}
from pprint import pprint
pprint(sampledIndices)
df
df.to_csv(f"./data.csv")

{0: array([374]),
 1: array([820]),
 2: array([801]),
 3: array([135]),
 4: array([486]),
 5: array([372])}


In [29]:
idx = 1
imageName = df.iloc[idx].name
s = re.search(r"ss-(\d+)-close-(\d+)-seq-(\d+).png", imageName)
nFrames = 5
seq = df[df.index.str.contains(f"ss-{int(s[1])}-close-{int(s[2])}-seq-")]
print(f"{len(seq) = }")

if len(seq) < nFrames:
      chosen = seq
else:
  i = random.randint(0, len(seq) - nFrames)
  chosen = seq[i: i + nFrames]
  
nMissing = nFrames - len(chosen)
if nMissing > 0:
  ## leave the zeros_like at the front
  chosen = ["empty" for _ in range(nMissing)] + chosen.index.to_list()
else: 
  chosen = chosen.index.to_list()
  
  
chosen    

len(seq) = 8


['ss-0-close-0-seq-1.png',
 'ss-0-close-0-seq-2.png',
 'ss-0-close-0-seq-3.png',
 'ss-0-close-0-seq-4.png',
 'ss-0-close-0-seq-5.png']

In [19]:
xs = [1,2,3,4,5,6, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
xs[-10:]

[8, 9, 10, 11, 12, 13, 14, 15, 16, 17]

In [40]:
df.head()
normd = df.copy()
dxs = normd["action"].map(lambda x: x[0])
dys = normd["action"].map(lambda x: x[1])

def normalise(x, ss):
  return 2 * ((x - ss.min()) / (ss.max() - ss.min())) - 1

normd["action"] = normd["action"].apply(lambda x: (normalise(x[0], dxs), normalise(x[1], dys)))
normd.head()



,agent_x,agent_y,target_x,target_y,action,closeness
ss-0-close-0.png,281,255,261,219,"(-0.4, -0.8)",0
ss-1-close-0.png,169,551,137,527,"(-0.8, -0.6)",0
ss-2-close-0.png,372,16,343,10,"(-0.8, -0.19999999999999996)",0
ss-3-close-0.png,89,395,62,377,"(-0.8, -0.4)",0
ss-4-close-0.png,758,197,747,175,"(-0.4, -0.8)",0


In [41]:
norm_name = "ss-info-normalised"
normd.to_pickle(f'./datasets/{folder}/{norm_name}.pkl')

In [27]:
with open("./datasets/1k-targets.pkl", "rb") as f:
    targets = pickle.load(f)
targets
  

[((350, 89, 589, 211), (4, 2)),
 ((354, 91, 589, 211), (4, 2)),
 ((358, 93, 589, 211), (4, 2)),
 ((362, 95, 589, 211), (4, 2)),
 ((366, 97, 589, 211), (4, 2)),
 ((370, 99, 589, 211), (4, 2)),
 ((374, 101, 589, 211), (4, 2)),
 ((378, 103, 589, 211), (4, 2)),
 ((382, 105, 589, 211), (4, 2)),
 ((386, 107, 589, 211), (4, 2)),
 ((390, 109, 589, 211), (4, 2)),
 ((394, 111, 589, 211), (4, 2)),
 ((398, 113, 589, 211), (4, 2)),
 ((402, 115, 589, 211), (4, 2)),
 ((406, 117, 589, 211), (4, 2)),
 ((410, 119, 589, 211), (4, 2)),
 ((414, 121, 589, 211), (4, 2)),
 ((418, 123, 589, 211), (4, 2)),
 ((422, 125, 589, 211), (4, 2)),
 ((426, 127, 589, 211), (4, 2)),
 ((430, 129, 589, 211), (4, 2)),
 ((434, 131, 589, 211), (4, 2)),
 ((438, 133, 589, 211), (4, 2)),
 ((442, 135, 589, 211), (4, 2)),
 ((446, 137, 589, 211), (4, 2)),
 ((450, 139, 589, 211), (4, 2)),
 ((454, 141, 589, 211), (4, 2)),
 ((458, 143, 589, 211), (4, 2)),
 ((462, 145, 589, 211), (4, 2)),
 ((466, 147, 589, 211), (4, 2)),
 ((470, 149, 589

In [37]:
fracs = {
  0: 0.25,
  1: 0.25,
  2: 1
}
gs = [df[df["closeness"] == c].sample(frac=frac) for c, frac in fracs.items()]
   

final = pd.concat(gs)
final



,agent_x,agent_y,target_x,target_y,action,closeness
ss-2-close-0.png,103,528,137,550,"(4, 2)",0
ss-0-close-1.png,507,541,419,359,"(-2, -4)",1
ss-0-close-2.png,257,12,642,515,"(3, 3)",2
ss-3-close-2.png,230,33,580,41,"(4, 0)",2
ss-1-close-2.png,56,107,371,233,"(4, 1)",2
ss-2-close-2.png,716,150,690,377,"(0, 4)",2


In [22]:
with open("./datasets/1k-targets-mirrored.pkl", "wb") as f:
    pickle.dump(mirrored, f)

In [3]:
import re
df["number"] = df.index.map(lambda s: int(re.search(r"ss-(\d+)-close-(\d+)-seq-(\d+).png", s)[1]))
df
# s = "ss-1-close-2-seq-3.png"
# cs = re.search(r"ss-(\d+)-close-(\d+)-seq-(\d+).png", s)
# cs[0]

closeness = {
      0: 0.001,
      1: 0.001,
      2: 0.001,
      3: 0.001,
      4: 0.001, ## 25
      5: 0.001, ##35
    }
sampledIndices = {c : np.random.choice(np.arange(0, 1000), size=int(frac * 1000)) for c, frac in closeness.items()}
from pprint import pprint
pprint(sampledIndices)
df

{0: array([533]),
 1: array([933]),
 2: array([570]),
 3: array([770]),
 4: array([38]),
 5: array([164])}


,agent_x,agent_y,target_x,target_y,action,closeness,number
ss-0-close-0-seq-0.png,460,319,375,250,"(-5, -5)",0,0
ss-0-close-0-seq-1.png,455,314,375,250,"(-5, -5)",0,0
ss-0-close-0-seq-2.png,450,309,375,250,"(-5, -5)",0,0
ss-0-close-0-seq-3.png,445,304,375,250,"(-5, -5)",0,0
ss-0-close-0-seq-4.png,440,299,375,250,"(-5, -5)",0,0
...,...,...,...,...,...,...,...
ss-999-close-5-seq-109.png,161,421,91,420,"(-5, 0)",5,999
ss-999-close-5-seq-110.png,156,421,91,420,"(-5, 0)",5,999
ss-999-close-5-seq-111.png,151,421,91,420,"(-5, 0)",5,999
ss-999-close-5-seq-112.png,146,421,91,420,"(-5, 0)",5,999


In [5]:
# print(df["number"] == 0)
gs = [df[(df["closeness"] == c) & (df["number"].isin(sampledIndices[c]))] for c, _ in closeness.items()]
together = pd.concat(gs)
print(together.shape)
together.to_csv(f"./togethersmall.csv")

(156, 7)


: 

In [63]:
from torch_bc import PositionDataset

transform = transforms.Compose([
    transforms.ToTensor()
])

ss = "./datasets/ss-1k"
img = f"{ss}/ss-0.png"

demo = Image.open(img) 
# print(f"Image size: {demo.size}")

# demo_img = transform(demo)
# import torchvision
plt.imshow(demo)
# print(demo.getbands())
# torchvision.transforms.functional.to_pil_image(demo_img, mode=None)
# demo_array = demo_img.numpy()*255
# print(Image.fromarray(demo_array.astype(np.uint8)))
# plt.imshow(demo_img, cmap='gray')


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\KUP\\Desktop\\Code\\fyp\\src\\2d\\datasets\\ss-1k\\ss-0.png'

In [ ]:
image = play_game(moveAgent="move_cnn", model=AgentNetwork_Regression())
plt.imshow(image)

In [ ]:
dataset[1]

In [ ]:
# import matplotlib.pyplot as plt
# from tqdm import tqdm as progress

# params = {
#   "datasize": [100, 200, 500, 700, 1000],
#   "epochs": [10, 50, 100, 200],
#   "lr": [0.001, 0.01, 0.1],
#   "batch_size": [32, 64, 128],
#   "neurons": [32, 64, 128, 256],
# }

# results = {}

# for size in params["datasize"]:
  
#   modelName = f"regression-{"1k" if size == 1000 else size}.pth"
#   with open("./datasets/1k-targets.pkl", "rb") as f:
#     data = pickle.load(f)

#   points = int(len(data) * (size / 1000))  
#   samples = random.sample(data, points)
#   for epoch in (params["epochs"]):
#     for lr in params["lr"]:
#       for batchSize in params["batch_size"]:
#         for neurons in params["neurons"]:
#           print(f"### size: {size}, epoch: {epoch}, lr: {lr}, batchSize: {batchSize}, neurons: {neurons}")
          
#           model = AgentNetwork_Regression(epochs=epoch, lr=lr, batchSize=batchSize, npl=neurons).train_on_behaviour(data=samples, overwriteDevice="cpu", doPrints=False)
#           results[(size, epoch, lr, batchSize, neurons)] = model.get_training_losses()


In [ ]:
results